# Semana 2: Programación orientada a objetos en Python

**Módulo 0** · Facultad de Ciencias, UNAM

## Objetivos de la sesión

1. Definir clases propias con atributos, métodos y `@property`.
2. Sobrecargar operadores con métodos especiales (`__repr__`, `__eq__`,
   `__add__`).
3. Distinguir objetos mutables de inmutables, y explicar por qué los
   objetos matemáticos deben ser inmutables.

## Antes de empezar

Esta clase sigue con el mismo entorno de la semana 1 — no hay nada nuevo
que instalar. Si no revisaste el checklist de
[`preparacion.md`](../preparacion/preparacion.md), hazlo ahora.

Vamos a construir una sola clase, `Vector2D`, **por capas**: cada sección
le agrega una capacidad y explica qué problema resuelve. Como es normal al
trabajar en un notebook, redefinimos la clase completa en cada paso; para
que cada celda quepa en pantalla, cada versión se queda solo con lo que
necesita la sección.

In [ ]:
import math

## De funciones a objetos

Hasta ahora representábamos una fuerza en el plano como una tupla de dos
números:

```python
fuerza = (3.0, 4.0)
magnitud = math.sqrt(fuerza[0]**2 + fuerza[1]**2)
```

Funciona, pero el dato y las operaciones que le corresponden viven
separados: nada impide pasarle esa tupla a una función que espera una
posición, y `fuerza[0]` no dice en ningún lado que sea la componente $x$.

Una **clase** junta las dos cosas —los datos (*atributos*) y lo que se
puede hacer con ellos (*métodos*)— bajo un nombre que dice qué es el
objeto.

## Clases y objetos: `__init__`, atributos y métodos

- La **clase** es el molde; el **objeto** (o *instancia*) es cada ejemplar
  construido con ese molde.
- `__init__` se ejecuta al construir la instancia: recibe los datos y los
  guarda como atributos.
- `self` es la instancia sobre la que se está trabajando. Es el primer
  parámetro de todo método de instancia, y Python lo pasa solo — no lo
  escribes al llamar el método.

In [ ]:
class Vector2D:
    """Vector en el plano, dado por sus componentes cartesianas."""

    def __init__(self, x, y):
        self.x = x   # atributos de instancia: cada vector tiene los suyos
        self.y = y

    def magnitud(self):
        return math.sqrt(self.x**2 + self.y**2)


v = Vector2D(3.0, 4.0)
v.magnitud()   # Vector2D es el molde; v es un objeto construido con él

## TODO en clase 1

Agrega a `Vector2D` un método `escalar(self, k)` que devuelva **un vector
nuevo** con ambas componentes multiplicadas por $k$:

$$k\,\vec{v} = (k\,v_x,\; k\,v_y)$$

Ojo con la palabra *nuevo*: el método no debe modificar `self`, sino
devolver un `Vector2D(...)` recién construido. Vamos a insistir mucho en
eso hoy.

In [ ]:
# TODO en clase: agrega el método escalar(self, k), que devuelve un Vector2D nuevo
class Vector2D:

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def magnitud(self):
        return math.sqrt(self.x**2 + self.y**2)

    def escalar(self, k):
        ...

## Instancia vs. clase; `@property`, `@classmethod`, `@staticmethod`

No todo lo que vive en una clase pertenece a cada instancia:

| Elemento | Recibe | Para qué sirve |
|---|---|---|
| Atributo de instancia (`self.x`) | — | Dato propio de cada objeto |
| Atributo de clase (`dimension = 2`) | — | Dato compartido por todas las instancias |
| Método de instancia | `self` | Opera sobre *este* objeto |
| `@property` | `self` | Método que se **usa como si fuera un atributo**: `v.magnitud`, sin paréntesis |
| `@classmethod` | `cls` | Constructor alternativo: otra forma de fabricar instancias |
| `@staticmethod` | nada | Función relacionada con la clase, que no necesita ni la instancia ni la clase |

`@property` es la que más vamos a usar. Sirve para exponer un valor
*calculado* con la misma sintaxis que un atributo guardado: quien usa el
objeto no necesita saber cuál de las dos cosas es.

In [ ]:
class Vector2D:

    dimension = 2   # atributo de clase: lo comparten todas las instancias

    def __init__(self, x, y):
        self.x = x
        self.y = y

    @property
    def magnitud(self):
        # Se calcula al momento, pero se lee como atributo: v.magnitud
        return math.sqrt(self.x**2 + self.y**2)

    @classmethod
    def desde_polares(cls, r, theta):
        # Constructor alternativo: fabrica un Vector2D a partir de (r, theta)
        return cls(r * math.cos(theta), r * math.sin(theta))


v = Vector2D(3.0, 4.0)
w = Vector2D.desde_polares(1.0, math.pi / 2)

v.magnitud, w.y, Vector2D.dimension   # magnitud, sin paréntesis

## TODO en clase 2

![Vector y el ángulo θ que forma con el eje x](img/vector-angulo.svg)

Agrega una `@property` llamada `angulo` que devuelva el ángulo del vector
respecto al eje $x$, en radianes.

Usa `math.atan2(self.y, self.x)`, **no** `math.atan(self.y / self.x)`:
`atan2` mira el signo de las dos componentes y por eso acierta el
cuadrante, además de no romperse cuando $v_x = 0$. Compruébalo con
`Vector2D(-1.0, 0.0)`: el ángulo debe ser $\pi$, no $0$.

In [ ]:
# TODO en clase: agrega la @property angulo, usando math.atan2
class Vector2D:

    def __init__(self, x, y):
        self.x = x
        self.y = y

    @property
    def magnitud(self):
        return math.sqrt(self.x**2 + self.y**2)

    @property
    def angulo(self):
        ...

## Métodos especiales (*dunder methods*)

Los métodos con doble guion bajo (*double underscore* → *dunder*) son la
forma en que una clase propia se engancha a la sintaxis de Python. No los
llamas tú: los llama el intérprete.

| Escribes | Python llama |
|---|---|
| `repr(v)`, o el resultado de una celda | `v.__repr__()` |
| `print(v)`, `str(v)` | `v.__str__()` (si falta, usa `__repr__`) |
| `v == w` | `v.__eq__(w)` |
| `v + w` | `v.__add__(w)` |
| `v * k` | `v.__mul__(k)` |
| `hash(v)`, `{v: ...}` | `v.__hash__()` |

Sin `__repr__`, el notebook muestra algo como
`<__main__.Vector2D object at 0x7f...>`, que no informa nada. Sin
`__eq__`, `Vector2D(1, 2) == Vector2D(1, 2)` resulta `False`: por defecto
Python compara **identidad** (si son el mismo objeto en memoria), no
contenido.

Convención para `__repr__`: devolver, cuando se pueda, el código que
reconstruye el objeto — `Vector2D(3.0, 4.0)`.

In [ ]:
class Vector2D:

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector2D({self.x}, {self.y})"

    def __eq__(self, otro):
        if not isinstance(otro, Vector2D):
            return NotImplemented   # deja que Python intente la operación inversa
        return self.x == otro.x and self.y == otro.y

    def __add__(self, otro):
        return Vector2D(self.x + otro.x, self.y + otro.y)


Vector2D(3.0, 4.0) + Vector2D(1.0, 1.0), Vector2D(1, 2) == Vector2D(1, 2)

## TODO en clase 3

Agrega dos métodos especiales más:

1. `__mul__(self, k)`, para que `v * 3` devuelva el vector escalado — el
   mismo cálculo del método `escalar` del TODO 1, ahora con sintaxis de
   operador.
2. `__sub__(self, otro)`, para que `v - w` funcione.

Los dos devuelven un `Vector2D` **nuevo**. Al terminar, comprueba que
`Vector2D(3.0, 4.0) * 2` da `Vector2D(6.0, 8.0)`.

In [ ]:
# TODO en clase: agrega __mul__ (por un escalar) y __sub__; ambos devuelven un Vector2D nuevo
class Vector2D:

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector2D({self.x}, {self.y})"

    def __eq__(self, otro):
        if not isinstance(otro, Vector2D):
            return NotImplemented
        return self.x == otro.x and self.y == otro.y

    def __add__(self, otro):
        return Vector2D(self.x + otro.x, self.y + otro.y)

    def __mul__(self, k):
        ...

    def __sub__(self, otro):
        ...

## Mutabilidad vs. inmutabilidad

Un objeto es **mutable** si su estado puede cambiar después de construido.
Nuestro `Vector2D` lo es: nada impide escribir `v.x = 99`.

Eso parece cómodo, y es la fuente de los errores más difíciles de rastrear
en cómputo científico. Si guardas un vector en una lista, en un
diccionario o dentro de otro objeto, y luego alguien lo modifica, el
cambio aparece en todos lados a la vez — porque todos comparten el mismo
objeto, no copias.

In [ ]:
v = Vector2D(3.0, 4.0)
fuerzas = [v, v]   # la misma referencia dos veces, no dos copias

v.x = 99.0         # modificamos "una"...

fuerzas            # ...y cambiaron las dos

In [ ]:
# El mismo objeto v, ahora como llave de un diccionario
try:
    {v: "algo"}
except TypeError as error:
    print("Tampoco puede ser llave de un diccionario:", error)

## Cómo se vuelve inmutable un objeto

El patrón estándar en Python: guardar el dato en un atributo *privado*
(`self._x`, por convención con un guion bajo inicial) y exponerlo con una
`@property` **sin** definir su `setter`. Al no haber setter, asignar
`v.x = 99` lanza `AttributeError`.

Con eso ganamos algo más. Un objeto que no cambia puede definir `__hash__`
de forma segura, y por lo tanto puede usarse como llave de diccionario o
elemento de un conjunto. Fíjate en el detalle: al definir `__eq__` en la
celda anterior, Python **desactivó** el `__hash__` por defecto de la clase,
justo para evitar que un objeto mutable acabe usándose como llave.

**Esta es la regla que siguen todos los objetos matemáticos de SymPy**, y
la razón por la que la vemos hoy: una expresión simbólica nunca se
modifica; las operaciones devuelven expresiones nuevas.

In [ ]:
class Vector2D:

    def __init__(self, x, y):
        self._x = x   # el guion bajo dice "no toques esto desde afuera"
        self._y = y

    @property
    def x(self):
        return self._x   # solo lectura: no definimos un setter

    @property
    def y(self):
        return self._y

    def __repr__(self):
        return f"Vector2D({self._x}, {self._y})"

    def __eq__(self, otro):
        if not isinstance(otro, Vector2D):
            return NotImplemented
        return self._x == otro._x and self._y == otro._y

    def __hash__(self):
        return hash((self._x, self._y))   # seguro: el estado ya no cambia

    def __add__(self, otro):
        return Vector2D(self._x + otro._x, self._y + otro._y)

In [ ]:
v = Vector2D(3.0, 4.0)

try:
    v.x = 99.0
except AttributeError as error:
    print("No se puede modificar:", error)

{Vector2D(1.0, 0.0): "eje x", Vector2D(0.0, 1.0): "eje y"}   # ahora sí es hashable

## Herencia y `super()`

Una clase puede construirse **a partir de otra**: hereda sus atributos y
métodos, y agrega o cambia solo lo que necesita. `super()` da acceso a la
implementación de la clase madre — típicamente para reusar su `__init__`
en vez de repetirlo.

Una fuerza *es* un vector en el plano, pero además sabe qué interacción
representa ("gravedad", "normal"). Ese **"es un"** es justamente la señal
de que la herencia aplica.

Así está construido SymPy: cada tipo de objeto matemático es una clase que
hereda de otra más general, y por eso todos comparten el mismo
comportamiento básico. Esa jerarquía la abrimos en la semana 12 — hoy
construimos una de dos niveles.

In [ ]:
class Fuerza(Vector2D):

    def __init__(self, x, y, nombre):
        super().__init__(x, y)   # reusa el __init__ de Vector2D
        self._nombre = nombre

    @property
    def nombre(self):
        return self._nombre

    def __repr__(self):
        return f"Fuerza({self._x}, {self._y}, {self._nombre!r})"


peso = Fuerza(0.0, -9.8, "gravedad")

peso, peso.y, isinstance(peso, Vector2D)   # la property y se heredó sin escribirla

## Polimorfismo y *duck typing*

**Polimorfismo**: el mismo código funciona con objetos de clases distintas,
y cada uno responde a su manera. Arriba ya ocurrió — al mostrar `peso`,
Python usó el `__repr__` de `Fuerza`, no el de `Vector2D`.

***Duck typing***: Python no exige que un objeto pertenezca a cierta clase,
solo que sepa hacer lo que se le pide. "Si camina como pato y grazna como
pato, es un pato". La función de abajo suma cualquier cosa que implemente
`__add__`; nunca pregunta de qué tipo es.

In [ ]:
def resultante(vectores):
    total = vectores[0]
    for siguiente in vectores[1:]:
        total = total + siguiente   # lo único que exige es que exista __add__
    return total


resultante([
    Fuerza(0.0, -9.8, "gravedad"),
    Fuerza(0.0, 9.8, "normal"),
    Vector2D(3.0, 0.0),
])

## Conexión con SymPy (demo motivacional)

Esto es solo una demostración — **no se espera que ustedes escriban código
todavía**. Empezamos con SymPy formalmente en la semana 4.

Todo lo de hoy es, literalmente, cómo está hecho SymPy por dentro: sus
objetos matemáticos son instancias de clases, con métodos especiales, y no
se pueden modificar. Veámoslo.

In [ ]:
import sympy as sp

x = sp.Symbol('x')

x + x   # Symbol define __add__, igual que nuestro Vector2D

In [ ]:
type(x), type(x + x)   # la suma no modifica x: devuelve un objeto nuevo, de otra clase

In [ ]:
# Inmutable, igual que nuestro Vector2D final: no admite atributos nuevos
try:
    x.etiqueta = "posición"
except AttributeError as error:
    print("No se le puede agregar nada:", error)

{x: 3.0}   # y como nunca cambia, es hashable: sirve de llave

Nada de eso es casualidad: `Symbol` es una clase, `x` es una instancia,
`__add__` está definido, y el objeto es inmutable y hashable por las
mismas razones que discutimos hoy. La jerarquía completa de clases sobre
la que está construido SymPy la abrimos en la **semana 12**, cuando les
toque crear sus propios objetos matemáticos.

## Resumen

Hoy construimos una clase propia por capas: atributos y `__init__`,
`@property` y `@classmethod`, métodos especiales para engancharse a la
sintaxis de Python (`__repr__`, `__eq__`, `__add__`, `__hash__`),
inmutabilidad con propiedades de solo lectura, y herencia con `super()`
más *duck typing*.

Es el ensayo general del Módulo 4: en la semana 12 volvemos a esto mismo,
pero heredando de las clases de SymPy en vez de las nuestras.

**Tarea de esta semana:** [`tarea-02.ipynb`](../tarea/tarea-02.ipynb) —
entrega antes de la clase de la Semana 3.

**Próxima clase — Semana 3:** Git y GitHub.